In [ ]:
import re

def extract_location(query):
    query = query.lower()

    # 1. Strip out common time patterns beforehand to clean the query
    time_patterns = [
        r"\bat\s\d{1,2}(?::\d{2})?\s*(?:am|pm|a\.m\.|p\.m\.)\b", # e.g., "at 8 am", "at 4:30 pm"
        r"\b\d{1,2}\s*(?:am|pm|a\.m\.|p\.m\.)\b",                 # e.g., "8am", "10 pm"
        r"\bin\ the\s(?:morning|afternoon|evening)\b"
    ]
    
    for tp in time_patterns:
        query = re.sub(tp, "", query)

    # 2. Use specific prepositions to isolate the location
    # We use a lazy match (.+?) followed by word boundaries to stop before next prepositions
    patterns = [
        r"\bnear\s+(.+?)(?:\s+(?:in|at|around)\b|$)",
        r"\bin\s+(.+?)(?:\s+(?:near|at|around)\b|$)",
        r"\bat\s+(.+?)(?:\s+(?:near|in|around)\b|$)"
    ]

    for p in patterns:
        match = re.search(p, query)
        if match:
            # Clean up residual whitespace or punctuation
            return match.group(1).strip().strip(",.-")

    # Fallback: If no prepositions matched, the whole query might just be the location
    return None

# Test runs
print(extract_location("Dermatologist near Erandwane Pune at 8 am"))  # Output: erandwane pune
print(extract_location("Pediatrician in Ravet around 4pm"))          # Output: ravet
print(extract_location("Dentist at Bhusawal tomorrow in the morning"))        # Output: bhusawal


erandwane pune
ravet
bhusawal tomorrow


In [47]:
import requests

def get_user_location():
    r = requests.get("https://ipapi.co/json/")
    data = r.json()
    print(f"User's location data: {data}")

    return {
        "lat": data["lat"],
        "lon": data["longitude"],
        "city": data["city"],
        "region": data["region"]
    }

print(get_user_location()) 

User's location data: {'error': True, 'reason': 'RateLimited', 'message': 'Visit https://ipapi.co/ratelimited/ for details'}


KeyError: 'lat'

In [ ]:
import json
import requests
import heapq
import math
from geopy.distance import geodesic

# Replace with your actual free SerpApi Key
SERP_API_KEY = "api_key"

In [4]:
def fetch_local_doctors(specialty, location):
    print(f"Querying SerpApi REST Endpoint for: '{specialty} clinic in {location}'...")
    example_location = (18.5010, 73.8326)
    # SerpApi's direct web routing parameters
    url = "https://serpapi.com/search"
    params = {
        "engine": "google_maps",
        "q": f"{specialty} clinic in {location}",
        "hl": "en",
        "api_key": SERP_API_KEY
    }

    try:
        # Fire standard HTTPS web call directly to the engine
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"❌ SerpApi server rejected query. Code: {response.status_code}")
            print(f"Server message: {response.text}")
            return []
        results = response.json()
        
        # Check if local map queries are returned in the payload array
        if "local_results" not in results:
            print("⚠️ No local listings found or API allocation credit exhausted.")
            return []
            
        raw_listings = results["local_results"]
        # print(json.dumps(raw_listings[0], indent=2))

        no_of_doctors=5
        distance_threshold=5    #in km
        #filtering list based on distance threshold
        valid_listings = [item for item in raw_listings if round(geodesic(example_location,(item.get("gps_coordinates", {}).get("latitude"), item.get("gps_coordinates", {}).get("longitude"))).km,2) <= distance_threshold]
        #sorting list based on ratings and reviews count
        top_n_listings = heapq.nlargest(no_of_doctors, valid_listings, key=lambda x: x.get("rating", 0) * math.log10(x.get("reviews", 0) + 1))
        ai_ready_doctors = []
        # Restrict parameters to top 5 hits to save context token fees in your LLM pipeline
        for item in top_n_listings:
                cleaned_profile = {
                    "name": item.get("title"),
                    "category": item.get("type") or specialty,
                    "phone": item.get("phone", "N/A"),
                    "address": item.get("address"),
                    "rating": f"{item.get('rating', 'N/A')} stars",
                    "reviews_count": item.get("reviews", 0),
                    "website": item.get("website", "None"),
                    "open_state": item.get("operating_hours", {}).get("open_now", "Unknown")}
                ai_ready_doctors.append(cleaned_profile)
        return ai_ready_doctors

    except Exception as e:
        print(f"Workflow execution pipeline failed: {e}")
        return []

# Run validation lookup trace
doctors_json_payload = fetch_local_doctors(specialty="Dermatologist", location="Erandwane, Pune")

print("\n🚀 Clean structured output for your AI Workflow:")
print(json.dumps(doctors_json_payload, indent=2))


Querying SerpApi REST Endpoint for: 'Dermatologist clinic in Erandwane, Pune'...

🚀 Clean structured output for your AI Workflow:
[
  {
    "name": "Oliva Skin, Hair & Laser Clinic Shivaji Nagar, Pune: Laser Hair Removal, Acne Scar, PRP, Skin Whitening Treatments",
    "category": "Dermatologist",
    "phone": "+91 89777 55434",
    "address": "Level 1, Deccan 99 Mall, No 1258, Jangali Maharaj Rd, opposite Deccan Avenue, Pulachi Wadi, Shivajinagar, Pune, Maharashtra 411004, India",
    "rating": "4.9 stars",
    "reviews_count": 1314,
    "website": "https://locations.olivaclinic.com/oliva-clinic/pune/shivaji-nagar/oliva-skin-hair-and-body-clinic-in-shivaji-nagar-pune--mSY32C/home",
    "open_state": "Unknown"
  },
  {
    "name": "Clear Skin",
    "category": "Dermatologist",
    "phone": "+91 95845 84111",
    "address": "CTC NO -94, 16 F.P NO -38/16, Prabhat Rd, Erandwane, Pune, Maharashtra 411004, India",
    "rating": "4.7 stars",
    "reviews_count": 1508,
    "website": "https:/

In [ ]:
import streamlit as st
import folium
from streamlit_folium import st_folium
import urllib.parse

st.title("Interactive Doctor Map")

# Start the map centered around the first doctor's location
start_lat = doctors_json_payload[0]["latitude"]
start_lon = doctors_json_payload[0]["longitude"]
m = folium.Map(location=[start_lat, start_lon], zoom_start=13)

# Add pins with clickable Google Maps redirection links inside HTML popups
for doc in doctors_json_payload:
    encoded_address = urllib.parse.quote(doc["address"])
    gmaps_url = f"https://google.com{encoded_address}"
    
    # Custom HTML layout inside the pin popup
    popup_html = f"""
    <h4>{doc['name']}</h4>
    <p><b>Rating:</b> {doc['rating']}</p>
    <p><b>Address:</b> {doc['address']}</p>
    <a href="{gmaps_url}" target="_blank" style="
        background-color: #4CAF50; 
        color: white; 
        padding: 5px 10px; 
        text-decoration: none; 
        border-radius: 4px;
        display: inline-block;
        margin-top: 5px;
    ">Get Directions</a>
    """
    
    folium.CircleMarker(
        location=[doc["latitude"], doc["longitude"]],
        radius=5,                      # Pixel size of circle radius
        # popup=folium.Popup(popup_html, max_width=250),
        tooltip=doc["name"],
        color="#FF4B4B",                # Border color
        fill=True,
        fill_color="#FF4B4B",           # Center fill color
        fill_opacity=1
        ).add_to(m)

# Render folium map in Streamlit
st_folium(m, width=500, height=500)


2026-06-24 10:52:48.223 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:52:48.225 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:52:48.227 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


KeyError: 'latitude'

In [8]:
import streamlit as st
import pandas as pd
import urllib.parse

# Sample data matching your ai_ready_doctors structure (with lat/lon added)
ai_ready_doctors = [
    {
        "name": "Dr. Smith (Cardiology)",
        "rating": "4.8 stars",
        "address": "123 Main St, New York, NY",
        "latitude": 40.7128,
        "longitude": -74.0060,
    },
    {
        "name": "Dr. Doe (Pediatrics)",
        "rating": "4.5 stars",
        "address": "456 Park Ave, New York, NY",
        "latitude": 40.7614,
        "longitude": -73.9776,
    }
]

st.title("Doctor Locator")

# 1. Convert to Pandas DataFrame for Streamlit Map
df = pd.DataFrame(ai_ready_doctors)

# 2. Render the Map (Streamlit looks for 'latitude' and 'longitude' columns)
st.map(df)

# 3. Handle Redirection below the map
st.subheader("Get Directions")
selected_doc = st.selectbox("Select a doctor to navigate:", df["name"])

# Find the selected doctor profile
doc_profile = next(item for item in ai_ready_doctors if item["name"] == selected_doc)

# Create a clean Google Maps search query link using the address
encoded_address = urllib.parse.quote(doc_profile["address"])
gmaps_url = f"https://google.com{encoded_address}"

# Streamlit link button for external redirection
st.link_button(f"Open Directions for {doc_profile['name']}", gmaps_url)


2026-06-24 10:54:32.182 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.184 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.185 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.257 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.259 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.260 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.261 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 10:54:32.262 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [ ]:
import urllib.parse

# Inside your loop or list comprehension:
address = item.get("address")
name = item.get("title")

# Create a safe, URL-encoded query for Google Maps directions
query_string = urllib.parse.quote(f"{doc["name"]}, {doc["address"]}" if doc["address"] else doc["name"])
gmap_link = f"https://google.com{query_string}"

# Add this to your cleaned profile dictionary:
# "gmap_link": gmap_link,
# "latitude": item.get("latitude"),   # Ensure your data fetches these float values
# "longitude": item.get("longitude")  # Required for Streamlit mapping


In [ ]:
#########################################################################################################

import streamlit as st
import pandas as pd

st.set_page_config(layout="wide")
st.title("👨‍⚕️ Top AI-Recommended Doctors")

# Mock data mimicking your 'ai_ready_doctors' output list
ai_ready_doctors = [
    {
        "name": "Dr. Smith (Cardiology)",
        "address": "123 Main St, New York, NY",
        "latitude": 40.7128,
        "longitude": -74.0060,
        "rating": "4.9 stars",
        "gmap_link": "https://google.com"
    },
    {
        "name": "Dr. Jones (Pediatrics)",
        "address": "456 Park Ave, New York, NY",
        "latitude": 40.7614,
        "longitude": -73.9680,
        "rating": "4.7 stars",
        "gmap_link": "https://google.com"
    }
]

# Convert listings to a DataFrame for Streamlit mapping
df = pd.DataFrame(ai_ready_doctors)

# Split screen into two columns: Map on left, Doctor Details on right
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Doctor Locations")
    # Streamlit automatically maps 'latitude' and 'longitude' columns
    st.map(df, size=20)

with col2:
    st.subheader("Doctor Directory")
    for doc in ai_ready_doctors:
        with st.container(border=True):
            st.markdown(f"### {doc['name']}")
            st.write(f"⭐ {doc['rating']}")
            st.write(f"📍 {doc['address']}")
            
            # Action button redirecting directly to Google Maps
            st.link_button("➡️ Get Directions on Google Maps", doc["gmap_link"])
